## Stockout Prediction Maps

This notebook draws the predictions of the production models on a map of the stations: the probability
that each station runs out of bikes (stockout) or out of docks (dockout, yes I realize this term is absurd) within the next 30 minutes.

`load_predictions` reads the predictions for a chosen New York time, and `plot_prediction_maps` (in `station_plots.py`)
draws them. The drawing function is shared with the Shiny app, which loads its data with SQL instead of Spark.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# Plotting functions shared with the EDA notebook and the Shiny app
from station_plots import plot_prediction_maps

spark.conf.set("spark.sql.session.timeZone", "UTC")

PREDICTIONS_TABLE = "workspace.default.station_predictions"
LOCAL_TZ = "America/New_York"

# Every station location, for the gray background of the maps
station_locations = (
    spark.table(PREDICTIONS_TABLE)
    .filter("lat IS NOT NULL AND lon IS NOT NULL")
    .select("lat", "lon").distinct()
    .toPandas()
)

In [0]:
# Which times have predictions? The times are shown in New York time.

display(
    spark.table(PREDICTIONS_TABLE)
    .agg(
        F.min(F.from_utc_timestamp("prediction_time", LOCAL_TZ)).alias("first_prediction_new_york"),
        F.max(F.from_utc_timestamp("prediction_time", LOCAL_TZ)).alias("last_prediction_new_york"),
        F.countDistinct("prediction_time").alias("prediction_times"),
    )
)

In [0]:
# load_predictions:
# Reads the predictions made at a New York time, for example "2026-10-08 08:00".
# Predictions are made every 5 minutes, so it takes the newest predictions at or before that time.
# With no time, it takes the newest predictions in the table.
# Returns the predictions as a pandas DataFrame, and their prediction time in New York time.

def load_predictions(new_york_time=None):
    all_predictions = spark.table(PREDICTIONS_TABLE)

    if new_york_time is not None:
        # prediction_time is stored in UTC, without a time zone
        utc_time = pd.Timestamp(new_york_time, tz=LOCAL_TZ).tz_convert("UTC").tz_localize(None)
        all_predictions = all_predictions.filter(F.col("prediction_time") <= F.lit(utc_time))

    prediction_time = all_predictions.agg(F.max("prediction_time")).first()[0]
    if prediction_time is None:
        raise ValueError(f"No predictions at or before {new_york_time}")

    station_predictions = (
        all_predictions
        .filter(F.col("prediction_time") == F.lit(prediction_time))
        .select("station_name", "lat", "lon",
                "num_bikes_available", "bike_state", "p_bike_stockout",
                "num_docks_available", "dock_state", "p_dock_stockout")
        .toPandas()
    )
    new_york_prediction_time = pd.Timestamp(prediction_time, tz="UTC").tz_convert(LOCAL_TZ)
    return station_predictions, new_york_prediction_time

### The newest predictions

In [0]:
station_predictions, new_york_prediction_time = load_predictions()
plot_prediction_maps(station_predictions, new_york_prediction_time, station_locations)
plt.show()

### Predictions at several times of one day

Change the times to any times that the table covers (see the cell near the top).

In [0]:
requested_times = [
    "2026-10-07 18:00",   # before the morning commute
    "2026-10-07 21:00",   # after the morning commute
    "2026-10-07 21:15",   # midday
    "2026-10-07 21:30",   # evening commute
]

for requested_time in requested_times:
    station_predictions, new_york_prediction_time = load_predictions(requested_time)
    plot_prediction_maps(station_predictions, new_york_prediction_time, station_locations)
    plt.show()